# Khảo sát MixLinear — C2: MixLinear + nhánh Linear

## Cấu hình của notebook này — C2, 992 tham số

```
(batch, 200) ─┬─ MixLinear (63 tham số)                      ─┐
              │                                               ├─ cộng ─ (batch, 25)
              └─ Linear(200,4) → Identity → Linear(4,25)  ────┘
```

Nhánh phụ nhận `x − mean(x)`, còn MixLinear tự trừ rồi cộng lại bên trong, nên
trung bình được cộng đúng **một** lần. Nhánh phụ nhìn dữ liệu ở thang biên độ
gốc, không chia độ lệch chuẩn.

Ghép là **phép cộng thuần**: không hệ số trộn học được, không gate, không norm,
không dropout. Train cả nền lẫn nhánh phụ cùng lúc từ đầu, không đóng băng gì.

C2 khác **C3** đúng một thứ: chỗ `Identity` kia thành `GELU`.

## Vì sao có khảo sát này

MixLinear-63 đã chạy: `cv_mean 0,672429 ± 0,006570`. Đường hội tụ cho thấy nó
**đã học hết** — epoch 18→19 chỉ còn giảm 0,28%, phẳng hơn cả TCN-64 — nhưng
dừng ở `train_mse 0,0500`, gấp 3,8 lần TCN. Là giới hạn **sức chứa**, không
phải thiếu epoch.

Mà MixLinear gần kịch trần thiết kế của chính nó: hai lớp `FLinear` nối nhau
không có phi tuyến nên hợp lại chỉ là một ma trận, hạng tối đa 3, đạt được từ
79 tham số. Thêm tham số vào nhánh cũ **không thể** thêm khả năng biểu diễn.
Muốn có sức chứa thật thì phải thêm đường đi mới.

## Thiết kế giai thừa — ba notebook chạy song song

|  | không nhánh phụ | có nhánh phụ |
|---|---|---|
| **không MixLinear** | — | **C0** (929) |
| **có MixLinear** | **B0** (63, đã chạy) | **C2** (992) |

Thêm **C3** (992): giống hệt C2, chỉ khác một `GELU`.

| notebook | model | tham số |
|---|---|---:|
| `TN_MixLinear_C0.ipynb` | `low_rank_linear` | 929 |
| `TN_MixLinear_C2.ipynb` | `mix_linear_linear` | 992 |
| `TN_MixLinear_C3.ipynb` | `mix_linear_mlp` | 992 |

Ba notebook **độc lập hoàn toàn**, mỗi cái nén ra tên zip riêng nên chạy cùng
lúc ở ba phiên Colab không đè nhau. Mỗi cái khoảng **45 phút**.

Bốn phép so đọc được sau khi cả ba xong:

| so sánh | trả lời câu gì | sạch không |
|---|---|---|
| **C3 − C2** | phi tuyến giúp gì | **sạch** — cùng 992 tham số, khác đúng GELU |
| **C2 − C0** | thêm MixLinear khi đã có nhánh phụ | lệch 63 tham số |
| **C2 − B0** | thêm nhánh phụ khi đã có MixLinear | lệch 929 tham số |
| C0 − B0 | 929 tham số tuyến tính so với 63 tham số MixLinear | đổi cả hai biến |

## Ba điều phải ghi khi báo cáo

**1. Nhánh phụ là đề xuất của đồ án, không phải MixLinear nguyên bản.** LSTNet
(SIGIR 2018) và N-BEATS (ICLR 2020) là tiền lệ cho việc ghép thành phần tuyến
tính với phi tuyến, **không** phải bằng chứng cho cấu hình chiều 4 trên UWB.

**2. C2 và C3 không bằng nhau tuyệt đối.** Ở C2, `bias` lớp đầu bị hấp thụ vào
`bias` lớp sau, vì hai `Linear` không phi tuyến hợp lại thành đúng một phép
affine. C2 có **988** tham số tác dụng độc lập so với 992 của C3. Chênh 0,4%,
không hỏng phép so, nhưng đừng viết "cùng hệt số tham số".

**3. Một seed là sàng lọc, không phải kiểm định.** B0 có `seed_std` **0,0066**,
ba seed trải 0,6648–0,6764. Chênh lệch dưới khoảng **0,007** trên một seed là
ngẫu nhiên. So với **`0,6760` của B0 seed 0**, không so với trung bình ba seed.

Và vì train chung từ đầu, **không** được nói nhánh phụ "học phần sai của
MixLinear" — nó chỉ là đường đi song song, hai bên học cùng nhau.

Thiết kế đầy đủ: `docs/THIET_KE_MIXLINEAR_DE_CLAUDE_REVIEW.md`

## 1. Chuẩn bị Colab

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Tải mã nguồn rồi vào thư mục đó.

In [ ]:
# Xoá trước để chạy lại ô này luôn lấy mã mới nhất, không dính bản cũ.
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

Lấy `by_user/` và `windows/` từ Drive.

In [ ]:
!python scripts/restore_processed_data_on_drive.py

## 2. Kiểm bản cài đặt

Chín phép kiểm. Bốn phép nhắm đúng chỗ dễ sai của thiết kế ghép nhánh:

    mục 6   gradient tới CẢ nền lẫn nhánh phụ — nếu một bên nằm chết thì
            kết quả nói về kiến trúc khác hẳn kiến trúc mình tưởng
    mục 7   xoá trọng số nhánh phụ, đầu ra phải khớp ĐÚNG MixLinear nền,
            chứng minh ghép là cộng thuần chứ không có gate hay hệ số ẩn
    mục 8   đưa vào chuỗi có trung bình 50, đầu ra phải quanh 50 chứ không
            phải 100 — trung bình chỉ được cộng một lần
    mục 9   Identity thật sự giữ nguyên giá trị, đúng vai trò đối chứng

In [ ]:
!python scripts/check_model.py --model mix_linear_linear

Xác nhận thêm một điều mà `check_model` chạy riêng lẻ không thấy được:
**C2 và C3 phải có cùng trọng số khởi tạo** khi cùng seed. Nếu khác thì phép so
`C3 − C2` lẫn cả chênh lệch khởi tạo, không còn cô lập được GELU.

`Identity` và `GELU` đều không có tham số và không tiêu thụ số ngẫu nhiên, nên
chỉ cần dựng module đúng thứ tự. Ô này chạy được ở cả hai notebook, cho cùng
kết quả — C3 cũng in dòng y hệt.

In [ ]:
import torch; from src import models, training
training.set_seed(0); d2 = models.build_model("mix_linear_linear").state_dict()
training.set_seed(0); d3 = models.build_model("mix_linear_mlp").state_dict()
print("cùng trọng số khởi tạo:", all(torch.equal(d2[k], d3[k]) for k in d2))

## 3. Chạy 4 fold, MỘT seed

Giao thức giữ nguyên: 20 epoch, Adam lr 1e-4, batch 64, MSE, `corr` 0,9, bốn
fold cũ. Chỉ đổi kiến trúc.

Tên cấu hình là `mix_linear_linear_p10_lpf5_h4_mse_corr0.9_seed0`, ghi vào
`runs/tn_mixlinear_ablation/`. Khoảng **45 phút** — 4,6 phút mỗi fold train
cộng khoảng 25 phút chấm điểm.

In [ ]:
!python scripts/run_cv.py --experiment tn_mixlinear_ablation --model mix_linear_linear --seed 0

## 4. Cất kết quả

Nén ra tên riêng `tn_mixlinear_c2.zip`, khác hai notebook kia, nên chạy song song
không đè nhau trên Drive.

In [ ]:
!python scripts/save_results.py tn_mixlinear_ablation --out tn_mixlinear_c2

## 5. Đường hội tụ

Cùng câu hỏi đã hỏi cho B0: điểm thấp là do hết sức chứa hay do 20 epoch chưa
đủ? B0 dừng ở `train_mse` **0,0500** và đã phẳng. Cấu hình này nhiều tham số
hơn nên đáng lẽ phải xuống thấp hơn — nếu không thì phần sức chứa thêm vào
không được dùng.

In [ ]:
!cut -d, -f1,2 runs/tn_mixlinear_ablation/mix_linear_linear_p10_lpf5_h4_mse_corr0.9_seed0_val_AB/curve.csv | tail -6

## 6. Bảng so

Chỉ thấy cấu hình của phiên này. Mốc để đặt cạnh:

| | tham số | cv_mean |
|---|---:|---:|
| **B0 MixLinear, seed 0** | 63 | **0,6760** |
| B0 MixLinear, 3 seed | 63 | 0,6724 ± 0,0066 |
| LSTM-67 | 56.908 | 0,7532 |

So **một seed với một seed**: dùng `0,6760`, không dùng trung bình ba seed. Và
nhớ ngưỡng nhiễu **0,007**.

Bảng gộp cả ba cấu hình dựng sau, khi đủ ba tệp zip trên Drive.

In [ ]:
!python scripts/compare_cv.py --experiment tn_mixlinear_ablation

## 7. Ngắt phiên

Kết quả đã nén sang Drive ở mục 4 nên ngắt ở đây không mất gì.

In [ ]:
from google.colab import runtime
runtime.unassign()